In [1]:
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_classic.document_loaders import TextLoader
from langchain_classic.schema import Document
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain


loader = TextLoader('/Users/pawanpahune/RAG_From_Scratch/Semantic_Chunking/langchain.txt')

embedding = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
chunker = SemanticChunker(embedding)
chunks = chunker.split_documents(loader.load())

for idx, chunk in enumerate(chunks):
    print(f"Chunk {idx+1}:\n{chunk.page_content}\n")
    print("-----")

new_document = """reinforcement learning is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize cumulative reward. 
It is inspired by behavioral psychology and is used in various applications such as robotics, game playing, and autonomous systems. 
the agent receives feedback in the form of rewards or penalties based on its actions, and it uses this feedback to improve its decision-making over time. 
Reinforcement learning algorithms include Q-learning, deep Q-networks (DQN), and policy gradient methods.
RL has successfully been applied to complex tasks like playing Go, controlling robots, and optimizing resource management. 
It continues to be an active area of research with ongoing advancements in algorithms and applications."""


new_doc = Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

chunks = chunker.split_documents([new_doc])



/Users/pawanpahune/RAG_From_Scratch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/ipykernel_39140/1261094174.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7682.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+-----------

Chunk 1:
Langchain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

-----
Chunk 2:
The Eiffel Tower is located in Paris. France is a popular tourist destination.

-----


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from langchain_classic.schema import Document
from langchain_classic.vectorstores import FAISS
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

vectorstore = FAISS.from_documents(chunks,embedding=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2'))
retriever  = vectorstore.as_retriever()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.7)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2876.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate


In [4]:
### create a prompt incluidng the conversation history

contextual_q_system_prompt = """
given a chat history and the latest user question which might reference context in the chat history, 
formulate a standalone question which can be understood without the chat history. 
If the latest user question does not reference any context in the chat history, 
DO NOT answer the question, just reformulate it if needed and otherwise return the question as is.
"""

In [5]:
contextual_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextual_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)


In [6]:
history_aware_retriever = create_history_aware_retriever(
     llm, retriever, contextual_q_prompt
)

In [7]:
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12fec1f90>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessag

In [8]:
qa_system_prompt = '''You are an assistant for question answering task.
use the following retrieved information to answer the question.
if you don't know the answer, say you don't know.
use three sentences at max to answer the question.

context:
{context}
'''

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

question_answer_chain = create_stuff_documents_chain(
    llm,
    qa_prompt
)

conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)

print("conversational_rag_chain ready")

conversational_rag_chain ready


In [9]:
chat_history = []

result1 = conversational_rag_chain.invoke(
    {
        "chat_history": chat_history,
        "input": "what is reinforcement learning?"
    }
)

print("Q: what is reinforcement learning?")
print(f"A: {result1['answer']}")

Q: what is reinforcement learning?
A: Reinforcement learning is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize cumulative reward. The agent receives feedback in the form of rewards or penalties based on its actions. It uses this feedback to improve its decision-making over time.
